# 05. Black-Scholes 공식 계산 원리 복습

이 노트북은 Black-Scholes 가격을 바로 암기하지 않고, `d1`, `d2`, 할인계수, 기초자산항, 할인 행사가항이 어떻게 연결되는지 확인하기 위한 복습용 노트다.

자세한 이론 설명은 `docs/black_scholes_review.md`와 연결되어 있으며, 반복 가능한 계산은 `src/black_scholes_review.py`에 분리되어 있다.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd

from src import MarketAssumption, ContractSpec, explain_black_scholes_price, build_black_scholes_review_table
from src.config import SAMPLE_DATA_DIR, TABLES_DIR

pd.options.display.float_format = '{:,.6f}'.format

## 1. 단일 ATM 콜 옵션을 손으로 따라가기

기준 시나리오 `S0=100, K=100, T=1, r=3%, sigma=20%`에서 콜 가격을 분해한다. 결과의 `stock_leg_value - bond_leg_value`가 콜 가격이 된다.

In [ ]:
market = MarketAssumption(spot=100.0, rate=0.03, volatility=0.20)
contract = ContractSpec(strike=100.0, maturity=1.0, option_type='call')

step = explain_black_scholes_price(
    market=market,
    contract=contract,
    scenario_id='notebook_atm_call',
    lesson_focus='ATM 콜 공식 분해',
)

pd.Series(step.to_dict())

## 2. 복습용 시나리오 전체 실행

`data/sample/black_scholes_review_scenarios.csv`는 ATM, ITM, OTM, 고변동성, 단기만기 사례를 작게 모아 둔 합성 입력이다. 실제 시장 데이터가 아니라 공식의 움직임을 비교하기 위한 예제다.

In [ ]:
scenario_path = SAMPLE_DATA_DIR / 'black_scholes_review_scenarios.csv'
scenarios = pd.read_csv(scenario_path)
scenarios

In [ ]:
review = build_black_scholes_review_table(scenarios)

review[[
    'scenario_id',
    'option_type',
    'd1',
    'd2',
    'stock_leg_value',
    'bond_leg_value',
    'price',
    'delta',
    'vega',
    'theta_per_day',
    'put_call_parity_gap',
]]

## 3. 결과 저장

반복 실행 가능한 산출물은 `outputs/tables/black_scholes_review_results.csv`에 저장한다. 커밋된 결과 테이블은 문서만 읽을 때도 공식 분해값을 확인할 수 있게 해준다.

In [ ]:
TABLES_DIR.mkdir(parents=True, exist_ok=True)
output_path = TABLES_DIR / 'black_scholes_review_results.csv'
review.to_csv(output_path, index=False)
output_path